# Lecture 9 — Implementing a Neural Network Library

**CMU 10-414/714, Deep Learning Systems (lecture by Tianqi Chen)**
Companion notebook · Sangram Lembe

Lecture 8 designed the pieces of a library — `Module`, `Parameter`, loss, optimizer. Lecture 9
builds them live, and on the way runs into two practical traps that have nothing to do with the
design and everything to do with making it actually work:

1. **the weight update itself leaks memory** if you write it the obvious way
2. **floating point is not real arithmetic** — `0.4` comes out as `0.39999992`, and a naive
   softmax returns `nan`

Both are reproduced below, then fixed. After that, the library pieces are assembled on the
lecture's own toy example, where every gradient can be checked by hand.

## 0. The tensor level

This engine mirrors needle more closely than the one in lecture 8, because this lecture depends on
two of needle's details:

- **`.data` returns a detached view that shares memory** — same numbers, no graph, no copy
- **if no input requires a gradient, the result is detached automatically** — no graph recorded

In [1]:
import numpy as np

class Tensor:
    """A needle-style tensor: data plus the op and inputs that produced it."""

    def __init__(self, array, requires_grad=True):
        self._init(None, [], np.asarray(array, dtype=np.float64), requires_grad)

    def _init(self, op, inputs, cached, requires_grad):
        self.op, self.inputs = op, list(inputs)
        self.cached, self.requires_grad, self.grad = cached, requires_grad, None

    @classmethod
    def make_const(cls, array):
        """A leaf with no history - what detach() returns."""
        t = Tensor.__new__(Tensor)
        t._init(None, [], array, False)
        return t

    @staticmethod
    def make_from_op(op, inputs):
        out = op.compute(*[i.cached for i in inputs])
        if not any(i.requires_grad for i in inputs):
            return Tensor.make_const(out)        # nothing needs a gradient: no graph
        t = Tensor.__new__(Tensor)
        t._init(op, inputs, out, True)
        return t

    # .data is a detached view that SHARES memory; assigning to it updates in place
    @property
    def data(self):
        return Tensor.make_const(self.cached)
    @data.setter
    def data(self, value):
        self.cached = value.cached if isinstance(value, Tensor) else np.asarray(value)

    def detach(self):  return self.data
    def numpy(self):   return self.cached
    shape = property(lambda s: s.cached.shape)
    def __repr__(self):
        return f"Tensor({np.array2string(self.cached, precision=4)}, op={type(self.op).__name__ if self.op else None})"

    def __add__(s, o):  return EAdd()(s, o) if isinstance(o, Tensor) else AddS(o)(s)
    def __mul__(s, o):  return EMul()(s, o) if isinstance(o, Tensor) else MulS(o)(s)
    def __neg__(s):     return MulS(-1.0)(s)
    def __sub__(s, o):  return s + (-o)
    def __rsub__(s, o): return (-s) + o
    def __pow__(s, c):  return PowS(c)(s)
    def __truediv__(s, o):
        return s * (o ** -1.0) if isinstance(o, Tensor) else MulS(1.0 / o)(s)
    def __matmul__(s, o): return MatMul()(s, o)
    __radd__, __rmul__ = __add__, __mul__

    def backward(self):
        backprop(self, Tensor.make_const(np.ones_like(self.cached)))


class Op:
    def __call__(self, *ins):
        return Tensor.make_from_op(self, ins)
    def grads(self, g, node):
        r = self.gradient(g, node)
        return r if isinstance(r, tuple) else (r,)

class EAdd(Op):
    def compute(s, a, b): return a + b
    def gradient(s, g, n): return g, g
class AddS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a + s.c
    def gradient(s, g, n): return g
class EMul(Op):
    def compute(s, a, b): return a * b
    def gradient(s, g, n): return g * n.inputs[1], g * n.inputs[0]
class MulS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a * s.c
    def gradient(s, g, n): return g * s.c
class PowS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a ** s.c
    def gradient(s, g, n): return g * (n.inputs[0] ** (s.c - 1)) * s.c
class MatMul(Op):
    def compute(s, a, b): return a @ b
    def gradient(s, g, n):
        return g @ transpose(n.inputs[1]), transpose(n.inputs[0]) @ g
class Transpose(Op):
    def compute(s, a): return a.T
    def gradient(s, g, n): return transpose(g)
class Reshape(Op):
    def __init__(s, shape): s.shape = shape
    def compute(s, a): return a.reshape(s.shape)
    def gradient(s, g, n): return reshape(g, n.inputs[0].shape)
class BroadcastTo(Op):
    def __init__(s, shape): s.shape = shape
    def compute(s, a): return np.broadcast_to(a, s.shape).copy()
    def gradient(s, g, n):                        # broadcast forward = sum backward
        ins = n.inputs[0].shape
        pad = len(s.shape) - len(ins)
        axes = tuple(range(pad)) + tuple(pad + i for i, d in enumerate(ins)
                                         if d == 1 and s.shape[pad + i] != 1)
        return reshape(summation(g, axes) if axes else g, ins)
class Summation(Op):
    def __init__(s, axes=None):
        s.axes = (axes,) if isinstance(axes, int) else axes
    def compute(s, a): return a.sum(axis=s.axes)
    def gradient(s, g, n):
        ins = n.inputs[0].shape
        ax = range(len(ins)) if s.axes is None else s.axes
        keep = tuple(1 if i in ax else d for i, d in enumerate(ins))
        return broadcast_to(reshape(g, keep), ins)
class ReLUOp(Op):
    def compute(s, a): return np.maximum(0, a)
    def gradient(s, g, n):
        return g * Tensor.make_const((n.inputs[0].cached > 0) * 1.0)
class Exp(Op):
    def compute(s, a): return np.exp(a)
    def gradient(s, g, n): return g * exp(n.inputs[0])
class LogSumExp(Op):
    """Stable log-sum-exp over axis 1: subtract the row max first."""
    def compute(s, a):
        m = a.max(1, keepdims=True)
        return (np.log(np.exp(a - m).sum(1, keepdims=True)) + m)[:, 0]
    def gradient(s, g, n):
        z = n.inputs[0]
        sm = exp(z - broadcast_to(reshape(n, (z.shape[0], 1)), z.shape))
        return broadcast_to(reshape(g, (z.shape[0], 1)), z.shape) * sm

transpose    = lambda a: Transpose()(a)
reshape      = lambda a, s: Reshape(s)(a)
broadcast_to = lambda a, s: BroadcastTo(s)(a)
summation    = lambda a, axes=None: Summation(axes)(a)
relu         = lambda a: ReLUOp()(a)
exp          = lambda a: Exp()(a)
logsumexp    = lambda a: LogSumExp()(a)

def topo_sort(root):
    """Post-order DFS, iterative so that very deep graphs cannot overflow the stack."""
    order, seen, stack = [], set(), [(root, False)]
    while stack:
        node, done = stack.pop()
        if done:
            order.append(node)
            continue
        if id(node) in seen:
            continue
        seen.add(id(node))
        stack.append((node, True))
        for i in node.inputs:
            if id(i) not in seen:
                stack.append((i, False))
    return order

def backprop(out, seed):
    """Reverse-mode AD: walk reverse topological order, summing partial adjoints."""
    parts = {id(out): [seed]}
    for node in reversed(topo_sort(out)):
        if id(node) not in parts: continue
        g = parts[id(node)][0]
        for extra in parts[id(node)][1:]:
            g = g + extra
        node.grad = g
        if node.op is not None:
            for inp, p in zip(node.inputs, node.op.grads(g, node)):
                if inp.requires_grad:
                    parts.setdefault(id(inp), []).append(p)

def numeric_grad(f, param, eps=1e-6):
    """Two-sided finite differences of scalar f() w.r.t. every entry of param."""
    g = np.zeros_like(param.cached)
    for idx in np.ndindex(*param.cached.shape):
        old = param.cached[idx]
        param.cached[idx] = old + eps; a = float(f().cached)
        param.cached[idx] = old - eps; b = float(f().cached)
        param.cached[idx] = old
        g[idx] = (a - b) / (2 * eps)
    return g

print("engine ready")

engine ready


## 1. The weight update leaks memory

Here is SGD written the obvious way. `w` requires gradients, so each line is recorded as a graph
node — and that node's input is the *previous* `w`.

In [2]:
w = Tensor(np.array([1.0, 1.0, 1.0]))            # requires_grad=True, like a real weight
g = Tensor(np.array([0.5, 0.5, 0.5]))
lr = 0.1

for step in range(5):
    w = w + (-lr) * g                            # the obvious update

print("w.op                     :", type(w.op).__name__)
print("w.inputs[0].op           :", type(w.inputs[0].op).__name__)
print("w.inputs[0].inputs[0].op :", type(w.inputs[0].inputs[0].op).__name__)
print("\nnodes reachable from w   :", len(topo_sort(w)))

w.op                     : EAdd
w.inputs[0].op           : EAdd
w.inputs[0].inputs[0].op : EAdd

nodes reachable from w   : 12


Follow `w.inputs` backwards and you walk the entire update history. None of it can be garbage
collected, because the newest `w` still points at it. Measure how that grows:

In [3]:
def graph_after(steps):
    w = Tensor(np.ones(3))
    for _ in range(steps):
        w = w + (-lr) * g
    return len(topo_sort(w))

print(f"{'steps':>8}{'nodes held in memory':>24}")
print("-" * 32)
for s in (1, 10, 100, 1000):
    print(f"{s:>8}{graph_after(s):>24}")

   steps    nodes held in memory
--------------------------------
       1                       4
      10                      22
     100                     202
    1000                    2002


Linear growth, forever. In a real training loop each of those nodes also holds on to whatever it
needed to compute its gradient, so this ends in an out-of-memory crash — one of the most common
bugs in imperative frameworks.

### The fix: `.data`

`w.data` returns a detached tensor. Three properties make it the right tool, and each can be
checked directly:

In [4]:
w = Tensor(np.array([1.0, 2.0, 3.0]))
d = w.data

print("d.op is None            :", d.op is None)
print("d.inputs                :", d.inputs)
print("d.requires_grad         :", d.requires_grad)
print("shares memory with w    :", np.shares_memory(d.cached, w.cached), "  <- no copy made")

d.op is None            : True
d.inputs                : []
d.requires_grad         : False
shares memory with w    : True   <- no copy made


In [5]:
# Arithmetic on detached tensors stays detached: no input needs a gradient.
new_w = w.data - g.data * lr
print("new_w.requires_grad :", new_w.requires_grad)
print("new_w.inputs        :", new_w.inputs)

# The right update: write back through the .data setter, in place.
w = Tensor(np.ones(3))
for _ in range(1000):
    w.data = w.data - g.data * lr
print("\nafter 1000 in-place updates, nodes held in memory:", len(topo_sort(w)))

new_w.requires_grad : False
new_w.inputs        : []

after 1000 in-place updates, nodes held in memory: 1


One node, however many steps. Writing back through `w.data = ...` matters for a second reason: the
weight usually lives inside a module. `w = ...` would create a new tensor that the module never
sees; updating in place changes the numbers the module already holds.

In [6]:
class Holder:                          # stands in for a module owning a weight
    def __init__(self): self.w = Tensor(np.ones(3))

m = Holder()
w = m.w                                # an optimizer holds a reference like this

w = w.data - g.data * lr               # rebinding: creates a NEW tensor
print("rebinding  -> module sees:", m.w.cached, "  (unchanged)")

w = m.w
w.data = w.data - g.data * lr          # in place: same object, new numbers
print("in place   -> module sees:", m.w.cached, "  (updated)")

rebinding  -> module sees: [1. 1. 1.]   (unchanged)
in place   -> module sees: [0.95 0.95 0.95]   (updated)


## 2. Floating point is not real arithmetic

Start at $1.0$ and subtract $0.1$ six times. In exact arithmetic that is $0.4$.

In [7]:
w = np.float32(1.0)
for _ in range(6):
    w = w + np.float32(-0.1)
print("float32 result:", repr(w))

from decimal import Decimal
print("0.1 as float32 is actually:", Decimal(float(np.float32(0.1))))

float32 result: np.float32(0.39999992)
0.1 as float32 is actually: 0.100000001490116119384765625


$0.1$ has no exact binary representation, so what gets stored is slightly off, and the error
accumulates. A float stores three fields in a fixed number of bits — sign, exponent, mantissa —
which can be pulled apart:

In [8]:
def float32_fields(x):
    bits = np.array(x, dtype=np.float32).view(np.uint32).item()
    sign     = bits >> 31
    exponent = (bits >> 23) & 0xFF
    mantissa = bits & 0x7FFFFF
    return sign, exponent - 127, 1 + mantissa / 2**23      # value = (-1)^s * m * 2^e

for x in (0.1, 0.4, 100.0):
    s, e, m = float32_fields(x)
    print(f"{x:>6} -> sign {s}, mantissa {m:.10f}, exponent {e:>3}"
          f"   reconstructed {(-1)**s * m * 2.0**e:.10f}")

fi = np.finfo(np.float32)
print(f"\nfloat32 largest value : {fi.max:.3e}")
print(f"e^100                 : {np.exp(100.0):.3e}   <- does not fit")

   0.1 -> sign 0, mantissa 1.6000000238, exponent  -4   reconstructed 0.1000000015
   0.4 -> sign 0, mantissa 1.6000000238, exponent  -2   reconstructed 0.4000000060
 100.0 -> sign 0, mantissa 1.5625000000, exponent   6   reconstructed 100.0000000000

float32 largest value : 3.403e+38
e^100                 : 2.688e+43   <- does not fit


### Where it bites: softmax

A naive softmax exponentiates first. In float32, $e^{100}$ is already infinite, and infinity
divided by infinity is `nan`.

The fix is an identity. Subtracting any constant $c$ from every input leaves softmax unchanged,
because the $e^{-c}$ cancels top and bottom:

$$\frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Choose $c = \max_j x_j$ and every exponent is at most zero.

In [9]:
def softmax_naive(x):
    z = np.exp(x)
    return z / z.sum()

def softmax_stable(x):
    z = np.exp(x - x.max())                # the identity, with c = max
    return z / z.sum()

for arr in ([0, 0, 1], [100, 100, 101], [1000, 1000, 1001]):
    x = np.array(arr, dtype=np.float32)
    with np.errstate(over="ignore", invalid="ignore"):
        naive = softmax_naive(x)
    print(f"{str(arr):<18} naive {np.round(naive, 4)}   stable {np.round(softmax_stable(x), 4)}")

[0, 0, 1]          naive [0.2119 0.2119 0.5761]   stable [0.2119 0.2119 0.5761]
[100, 100, 101]    naive [nan nan nan]   stable [0.2119 0.2119 0.5761]
[1000, 1000, 1001] naive [nan nan nan]   stable [0.2119 0.2119 0.5761]


The same trick gives a stable **log-sum-exp**, which is what the engine's `LogSumExp` op uses —
and why a loss built on it never forms a softmax explicitly.

In [10]:
x = np.array([[1000.0, 1000.0, 1001.0]])
with np.errstate(over="ignore"):
    naive = np.log(np.exp(x).sum())
print("naive log-sum-exp :", naive)
print("stable (engine)   :", logsumexp(Tensor(x)).cached)

naive log-sum-exp : inf
stable (engine)   : [1001.55144471]


## 3. `Parameter` and `Module`

A `Parameter` is a tensor subclass with nothing special about it — it only marks itself as
something to train. A `Module` finds its parameters by walking its own attributes recursively,
exactly as in the lecture's `_get_params`: parameters are collected, dictionaries and lists are
searched, and sub-modules are asked for theirs.

In [11]:
class Parameter(Tensor):
    """Nothing special - just a marker that this tensor is trainable."""

def _get_params(value):
    if isinstance(value, Parameter):
        return [value]
    if isinstance(value, dict):
        out = []
        for v in value.values():
            out += _get_params(v)
        return out
    if isinstance(value, (list, tuple)):
        out = []
        for v in value:
            out += _get_params(v)
        return out
    if isinstance(value, Module):
        return value.parameters()
    return []

class Module:
    def parameters(self):
        return _get_params(self.__dict__)      # every attribute of this object
    def __call__(self, *args):
        return self.forward(*args)

### The lecture's toy modules

`ScaleAdd` computes $s \cdot x + b$. `MultiPathScaleAdd` runs two of them side by side and adds the
results. It owns no parameters directly — only two sub-modules — which makes it a good test of the
recursion.

In [12]:
class ScaleAdd(Module):
    def __init__(self, init_s=1.0, init_b=0.0):
        self.s = Parameter(np.array([init_s]))
        self.b = Parameter(np.array([init_b]))
    def forward(self, x):
        return x * self.s + self.b

class MultiPathScaleAdd(Module):
    def __init__(self):
        self.path0 = ScaleAdd()
        self.path1 = ScaleAdd()
    def forward(self, x):
        return self.path0(x) + self.path1(x)

mpath = MultiPathScaleAdd()
params = mpath.parameters()
print("parameters found :", len(params))
print("params[0] is mpath.path0.s :", params[0] is mpath.path0.s)
print("params[3] is mpath.path1.b :", params[3] is mpath.path1.b)

parameters found : 4
params[0] is mpath.path0.s : True
params[3] is mpath.path1.b : True


Four parameters, found through two levels of nesting, and they are the *same objects* the
sub-modules hold — which is what lets an optimizer update them in place.

## 4. A loss module, and a gradient you can check by hand

A loss is a module too — tensor in, scalar out, no parameters of its own.

In [13]:
class L2Loss(Module):
    def forward(self, pred, y):
        diff = pred - y
        return summation(diff * diff)

x = Tensor(np.array([2.0]), requires_grad=False)
y = Tensor(np.array([2.0]), requires_grad=False)

pred = mpath(x)
loss = L2Loss()(pred, y)
loss.backward()

print("prediction :", pred.cached)
print("loss       :", loss.cached)
print("d loss / d path0.s :", mpath.path0.s.grad.cached)

prediction : [4.]
loss       : 4.0
d loss / d path0.s : [8.]


By hand, with every $s = 1$, $b = 0$, $x = 2$, $y = 2$:

$$\text{pred} = (1\cdot 2 + 0) + (1\cdot 2 + 0) = 4, \qquad
\ell = (4 - 2)^2 = 4, \qquad
\frac{\partial\ell}{\partial s_0} = 2(4 - 2)\cdot x = 8$$

Autodiff agrees. Checking all four gradients against finite differences as well:

In [14]:
f = lambda: L2Loss()(mpath(x), y)
for name, p in [("path0.s", mpath.path0.s), ("path0.b", mpath.path0.b),
                ("path1.s", mpath.path1.s), ("path1.b", mpath.path1.b)]:
    print(f"{name}: autodiff {p.grad.cached[0]:5.2f}   numeric {numeric_grad(f, p)[0]:5.2f}")

path0.s: autodiff  8.00   numeric  8.00
path0.b: autodiff  4.00   numeric  4.00
path1.s: autodiff  8.00   numeric  8.00
path1.b: autodiff  4.00   numeric  4.00


## 5. The optimizer

An optimizer needs exactly two methods:

- **`reset_grad`** — clear every parameter's gradient before the next backward pass
- **`step`** — apply the update, using detached arithmetic and in-place write-back from section 1

Momentum adds one thing: a zero tensor per parameter, created and owned by the optimizer.

In [15]:
class Optimizer:
    def __init__(self, params):
        self.params = params
    def reset_grad(self):
        for p in self.params:
            p.grad = None

class SGD(Optimizer):
    def __init__(self, params, lr=0.01):
        super().__init__(params)
        self.lr = lr
    def step(self):
        for w in self.params:
            w.data = w.data - w.grad.data * self.lr          # detached, in place

class SGDMomentum(Optimizer):
    def __init__(self, params, lr=0.01, beta=0.9):
        super().__init__(params)
        self.lr, self.beta = lr, beta
        self.u = [Tensor.make_const(np.zeros_like(p.cached)) for p in params]   # owned state
    def step(self):
        for i, w in enumerate(self.params):
            self.u[i] = self.u[i] * self.beta + w.grad.data * (1 - self.beta)
            w.data = w.data - self.u[i] * self.lr

## 6. The end-to-end loop

Every piece now plugs together. The target is $y = 7$ at $x = 2$.

In [16]:
def train(model, opt, loss_fn, x, y, epochs):
    history = []
    for _ in range(epochs):
        opt.reset_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        history.append(float(loss.cached))
    return history

x = Tensor(np.array([2.0]), requires_grad=False)
y = Tensor(np.array([7.0]), requires_grad=False)

m1 = MultiPathScaleAdd()
h_sgd = train(m1, SGD(m1.parameters(), lr=0.02), L2Loss(), x, y, epochs=30)

m2 = MultiPathScaleAdd()
h_mom = train(m2, SGDMomentum(m2.parameters(), lr=0.05, beta=0.9), L2Loss(), x, y, epochs=30)

print(f"{'epoch':>6}{'SGD loss':>12}{'momentum loss':>16}")
print("-" * 34)
for e in (0, 1, 2, 5, 10, 20, 29):
    print(f"{e:>6}{h_sgd[e]:>12.5f}{h_mom[e]:>16.5f}")
print("\nfinal prediction (SGD):", m1(x).cached, "  target 7.0")
print("graph nodes held by path0.s after training:", len(topo_sort(m1.path0.s)))

 epoch    SGD loss   momentum loss
----------------------------------
     0     9.00000         9.00000
     1     3.24000         7.29000
     2     1.16640         4.66560
     5     0.05442         0.00765
    10     0.00033         3.12006
    20     0.00000         1.06916
    29     0.00000         0.42018

final prediction (SGD): [6.99999934]   target 7.0
graph nodes held by path0.s after training: 1


SGD's loss falls smoothly, the prediction reaches the target, and after thirty steps each parameter
is still a single graph node — the in-place update kept memory flat.

Momentum does something worth noticing. It reaches a lower loss than SGD by epoch 5, then climbs back
up and oscillates. That is momentum's characteristic *overshoot*: the velocity it has built up
carries the parameters past the minimum, and on a problem this small and this well-behaved there is
nothing to gain from that extra speed. Plain SGD is close to ideal here. The point of the swap is the
engineering, not the ranking: changing the optimizer was one line, and the model, loss and loop were
untouched.

## 7. Initialisation, when the distribution is uniform

The lecture's initialisation rule is stated for Gaussians: variance $2/n$ for ReLU. For a uniform
distribution you need the variance *of the uniform* to match. A uniform on $[-a, a]$ has variance
$a^2/3$, so $a^2/3 = 2/n$ gives $a = \sqrt{6/n}$. Checking that it keeps activations the same size
through a deep ReLU stack:

In [17]:
def depth_norms(sampler, depth=30, n=256, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(1, n))
    for _ in range(depth):
        z = np.maximum(0, z @ sampler(rng, n))
    return np.linalg.norm(z)

gauss   = lambda rng, n: rng.normal(0, np.sqrt(2 / n), (n, n))
uniform = lambda rng, n: rng.uniform(-np.sqrt(6 / n), np.sqrt(6 / n), (n, n))
naive_u = lambda rng, n: rng.uniform(-np.sqrt(2 / n), np.sqrt(2 / n), (n, n))

print(f"Gaussian, var 2/n             : norm after 30 layers = {depth_norms(gauss):10.3g}")
print(f"uniform, bound sqrt(6/n)      : norm after 30 layers = {depth_norms(uniform):10.3g}")
print(f"uniform, bound sqrt(2/n) WRONG: norm after 30 layers = {depth_norms(naive_u):10.3g}")

Gaussian, var 2/n             : norm after 30 layers =       16.1
uniform, bound sqrt(6/n)      : norm after 30 layers =       8.74
uniform, bound sqrt(2/n) WRONG: norm after 30 layers =   6.09e-07


The correct uniform bound behaves like the Gaussian. Reusing the Gaussian's number as a uniform
bound gives variance $2/(3n)$ instead of $2/n$, and the activations collapse.

## Summary

- `w = w - lr * g` records a graph node every step, so the weight drags its whole history behind it.
- `w.data` is detached and **shares memory**, and arithmetic on detached tensors stays detached.
  Writing back through `w.data = ...` keeps memory flat *and* keeps the module's reference valid.
- Floats are sign × mantissa × 2^exponent in fixed bits, so $1 - 6\times0.1 = 0.39999992$ in
  float32, and $e^{100}$ overflows. Stable softmax and log-sum-exp subtract the max first.
- `Parameter` is only a marker; `Module.parameters()` finds them recursively through modules,
  dicts and lists, returning the same objects the modules hold.
- An optimizer is `reset_grad` plus `step`, and owns any extra state such as momentum.
- For uniform initialisation, match the *variance*: bound $\sqrt{6/n}$, not $\sqrt{2/n}$.